# R from SpenderQ reconstructions

Compare the same emission-line pairs from a completed Nσ run on `final_subset.csv`. The starting pairs already passed the raw-coadd peak Nσ threshold. This notebook repeats the line-flux cuts on the reconstructions.

The normalized version uses `RECON_FLUX`. The unnormalized version multiplies it by that epoch’s `NORM` and scales the noise with it. The uncertainties come from saved coadd weights and approximate the noise in this comparison.

In [1]:
from functools import lru_cache
from pathlib import Path
import json

from astropy.io import fits
from extinction import fitzpatrick99
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Existing Nσ results and the catalog to compare them against.
NSIGMA_DIR = Path("/work/10579/prisha/ls6/desi_project/nsigma_results")
RECON_DIR = Path("/work/10579/prisha/ls6/desi_project/reconstructions")
CHI2_CSV = Path("/work/10579/prisha/ls6/desi_project/post_chi2_calc.csv")
FINAL_CSV = Path("/work/10579/prisha/ls6/desi_project/final_subset.csv")

with (NSIGMA_DIR / "run_info.json").open() as file:
    settings = json.load(file)["arguments"]

key_columns = ["TARGETID", "NIGHT", "TILEID", "PETAL_LOC"]


def read_coadd_keys(path):
    keys = pd.read_csv(path, usecols=["TARGETID", "LASTNIGHT", "TILEID", "PETAL_LOC"],
                       dtype={"TARGETID": "string", "LASTNIGHT": "string"})
    keys["NIGHT"] = keys["LASTNIGHT"].str[:10].str.replace("-", "", regex=False)
    return keys[key_columns].drop_duplicates()


final_keys = pd.MultiIndex.from_frame(read_coadd_keys(FINAL_CSV))
peaks = pd.read_csv(NSIGMA_DIR / "nsigma_passed_line_metrics.csv",
                    dtype={"TARGETID": "string", "NIGHT1": "string", "NIGHT2": "string"})
before = len(peaks)
for i in (1, 2):
    keys = peaks[["TARGETID", f"NIGHT{i}", f"TILE{i}", f"PETAL{i}"]].copy()
    keys.columns = key_columns
    peaks = peaks.loc[pd.MultiIndex.from_frame(keys).isin(final_keys)]

print("Saved Nσ input:", settings["input_csv"])
print(f"Saved line-pair rows with both coadds in final_subset.csv: {len(peaks):,} / {before:,}")
print(f"Unique targets among those rows: {peaks['TARGETID'].nunique():,}")

# Identify ambiguous reconstruction filenames using the pre-cut coadd catalog.
coadd_keys = read_coadd_keys(CHI2_CSV)
shared = coadd_keys.duplicated(["TARGETID", "NIGHT"], keep=False)
ambiguous = set(coadd_keys.loc[shared, ["TARGETID", "NIGHT"]].itertuples(index=False, name=None))

WINDOWS = {
    "Lyalpha": ((1190, 1250), (1150, 1190), (1250, 1290)),
    "SiIV_OIV_blend": ((1360, 1430), (1310, 1360), (1430, 1480)),
    "CIV": ((1510, 1590), (1460, 1510), (1590, 1640)),
    "CIII_1909": ((1850, 1970), (1820, 1850), (1970, 2000)),
    "MgII": ((2750, 2850), (2680, 2720), (2880, 2920)),
}
step = settings["rest_bin_width"]
n_bins = round((settings["rest_max"] - settings["rest_min"]) / step)
edges = settings["rest_min"] + np.arange(n_bins + 1) * step
wave = (edges[:-1] + edges[1:]) / 2

Saved Nσ input: subset_cands.csv
Saved line-pair rows with both coadds in final_subset.csv: 111,069 / 125,154
Unique targets among those rows: 10,619


## Read the reconstructions and measure the line

The FITS header redshift restores the observed wavelength grid before shifting to the pair redshift. Apply the same Galactic extinction correction, bin widths, and line windows used in the Nσ run. Skip pairs that include a night with multiple distinct coadds because the reconstruction filename is shared.

In [2]:
def pixel_edges(centers):
    mid = (centers[:-1] + centers[1:]) / 2
    return np.r_[2 * centers[0] - mid[0], mid, 2 * centers[-1] - mid[-1]]


def rebin(native_wave, flux, ivar):
    native = pixel_edges(native_wave)
    good = np.isfinite(flux) & np.isfinite(ivar) & (ivar > 0)
    out = np.full(len(wave), np.nan)
    variance = out.copy()
    for j, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        start = max(0, np.searchsorted(native, lo, side="right") - 1)
        stop = min(len(flux), np.searchsorted(native, hi, side="left"))
        k = np.arange(start, stop)
        k = k[good[k]]
        w = (np.minimum(native[k + 1], hi) - np.maximum(native[k], lo)) / (hi - lo)
        if np.isclose(w.sum(), 1, rtol=0, atol=1e-8):
            out[j] = w @ flux[k]
            variance[j] = (w**2) @ (1 / ivar[k])
    return out, variance


@lru_cache(maxsize=32)
def read_reconstruction(tid, night, pair_z, ebv, rv):
    path = RECON_DIR / tid / night / f"recon_{tid}_{night}.fits"
    with fits.open(path) as hdul:
        hdu = hdul[1]
        observed = np.ascontiguousarray(hdu.data["REST_WAVE"], dtype=float) * (1 + hdu.header["Z"])
        flux = np.array(hdu.data["RECON_FLUX"], dtype=float)
        ivar = np.array(hdu.data["UPDATED_WEIGHTS"], dtype=float)
        norm = float(hdu.header["NORM"])
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError(f"Invalid NORM in {path}")
    dust = 10 ** (0.4 * fitzpatrick99(observed, rv * ebv, r_v=rv, unit="aa"))
    rebinned, variance = rebin(observed / (1 + pair_z), flux * dust, ivar / dust**2)
    return rebinned, variance, norm


def measure(flux, var, shared, windows):
    (lo, hi), left, right = windows
    blue = shared & (wave >= left[0]) & (wave <= left[1]) & (wave < lo)
    red = shared & (wave >= right[0]) & (wave <= right[1]) & (wave > hi)
    widths = np.maximum(0, np.minimum(edges[1:], hi) - np.maximum(edges[:-1], lo))
    line = shared & (wave >= lo) & (wave <= hi) & (widths > 0)
    if min(blue.sum(), red.sum(), line.sum()) < 2:
        raise ValueError("Insufficient shared valid pixels in the line or continuum sidebands")

    side = blue | red
    design = np.column_stack([np.ones(len(wave)), (wave - (lo + hi) / 2) / 100])
    sigma = np.sqrt(var[side])
    beta = np.linalg.lstsq(design[side] / sigma[:, None], flux[side] / sigma, rcond=None)[0]
    continuum = design @ beta
    integrated = widths[line] @ (flux[line] - continuum[line])
    error = np.sqrt((widths[line]**2) @ var[line])
    return integrated, error, continuum, line


def r_metrics(measurements):
    # Sort by integrated flux, since undoing the normalization can change the order.
    (dim, dim_err), (bright, bright_err) = sorted(measurements)
    if dim <= 0:
        return np.nan, np.nan, np.nan
    ratio = bright / dim
    r = ratio - 1
    sigma_r = np.hypot(bright_err / dim, ratio * dim_err / dim)
    return r, sigma_r, dim / dim_err


def pair_data(row):
    nights = [str(int(row["NIGHT1"])), str(int(row["NIGHT2"]))]
    obs = [read_reconstruction(row["TARGETID"], night, row["Z_USED"], row["EBV_GAL"], row["RV_GAL"])
           for night in nights]
    shared = np.logical_and.reduce([
        np.isfinite(f) & np.isfinite(v) & (v > 0) for f, v, _ in obs
    ])
    measured = [measure(f, v, shared, WINDOWS[row["LINE"]]) for f, v, _ in obs]
    return obs, measured

## Count targets left after the R cuts

A target survives a stage if any one of its lines and pairs survives that stage. The raw-coadd column uses the same pairs with usable reconstructions for a like-for-like comparison.

In [3]:
records = []
for i, row in peaks.iterrows():
    record = {"row": i, "TARGETID": row["TARGETID"]}
    nights = [str(int(row["NIGHT1"])), str(int(row["NIGHT2"]))]
    if any((row["TARGETID"], night) in ambiguous for night in nights):
        record["status"] = "Ambiguous same-night reconstruction"
    else:
        try:
            obs, measured = pair_data(row)
            normalized = [(f, e) for f, e, _, _ in measured]
            # A constant multiplier scales the integrated flux and its error equally.
            unnormalized = [(f * norm, e * norm) for (f, e), (_, _, norm) in zip(normalized, obs)]
            record["normalized"] = r_metrics(normalized)
            record["unnormalized"] = r_metrics(unnormalized)
            record["status"] = "Measured"
        except FileNotFoundError:
            record["status"] = "Missing reconstruction FITS"
        except (KeyError, ValueError, OSError, np.linalg.LinAlgError) as error:
            record["status"] = type(error).__name__ + ": " + str(error).split(" in ")[0]
    records.append(record)

results = pd.DataFrame.from_records(records, columns=["row", "TARGETID", "status",
                                                     "normalized", "unnormalized"])
display(results.groupby("status").agg(line_pairs=("row", "size"),
                                      targets=("TARGETID", "nunique")))
usable = results.loc[results["status"].eq("Measured")].set_index("row")
raw = peaks.loc[usable.index]
threshold = settings["r_min"]
significance = settings["r_sigma_factor"]


def count_stages(values):
    r, sigma_r, dim_snr = np.asarray(values, dtype=float).reshape(-1, 3).T
    finite = np.isfinite(r) & np.isfinite(sigma_r) & np.isfinite(dim_snr)
    amplitude = finite & (r > threshold)
    significant = amplitude & (r > significance * sigma_r)
    final = significant & (dim_snr < 2 * r)
    return [usable.loc[keep, "TARGETID"].nunique() for keep in
            (np.ones(len(usable), dtype=bool), finite, amplitude, significant, final)]


counts = pd.DataFrame({
    "Raw coadd": count_stages(raw[["R", "SIGMA_R", "F_SIGMA_DIM"]].to_numpy()),
    "Normalized recon": count_stages(usable["normalized"].tolist()),
    "Unnormalized recon": count_stages(usable["unnormalized"].tolist()),
}, index=["Targets with usable reconstructions", "Finite R and uncertainty",
          f"R > {threshold:g}", f"Also R > {significance:g} × σR", "Also dim flux / error < 2R"])
print(f"Raw Nσ starting cohort: {peaks['TARGETID'].nunique():,} unique targets")
display(counts)

,line_pairs,targets
status,,
Ambiguous same-night reconstruction,489,25
Measured,80540,10084
ValueError: Insufficient shared valid pixels,30040,7461


Raw Nσ starting cohort: 10,619 unique targets


,Raw coadd,Normalized recon,Unnormalized recon
Targets with usable reconstructions,10084,10084,10084
Finite R and uncertainty,10021,10076,10076
R > 2,562,102,107
Also R > 2 × σR,29,11,5
Also dim flux / error < 2R,23,8,5


## Plot one comparison

The dashed curves are the fitted continua. The shaded area between each curve and its spectrum is the signed line flux. The full spectra are above the zoomed line panels.

In [4]:
if usable.empty:
    raise ValueError("No usable reconstruction pairs. Check the status table above.")

# Change this index to show another measurable line and pair.
row = peaks.loc[usable.index[0]]
obs, measured = pair_data(row)
(lo, hi), left, right = WINDOWS[row["LINE"]]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
for column, mode in enumerate(("normalized", "unnormalized")):
    full, zoom = axes[:, column]
    full.axvspan(lo, hi, color="gray", alpha=0.18)
    zoom.axvspan(*left, color="gray", alpha=0.15)
    zoom.axvspan(*right, color="gray", alpha=0.15)
    for i, ((flux, _, norm), (line_flux, _, continuum, line)) in enumerate(zip(obs, measured)):
        scale = 1 if mode == "normalized" else norm
        color = ("#0072B2", "#D55E00")[i]
        full.plot(wave, flux * scale, color=color, lw=0.8, label=row[f"OBS{i+1}"])
        zoom.plot(wave, flux * scale, color=color, label=f"Line flux: {line_flux * scale:.3g}")
        zoom.plot(wave, continuum * scale, "--", color=color)
        zoom.fill_between(wave, continuum * scale, flux * scale, where=line,
                          step="mid", color=color, alpha=0.25)
    r = usable.at[row.name, mode][0]
    full.set_title(f"{mode.capitalize()} reconstruction · R = {r:.3g}")
    zoom.set_xlim(left[0], right[1])
    for ax in (full, zoom):
        ax.set(xlabel="Rest-frame wavelength (Å)", ylabel="Flux density")
        ax.legend(fontsize=8)
fig.suptitle(f"{row['TARGETID']} · {row['LINE']}")
plt.show()